In [ ]:

# ЯЧЕЙКА 1

!pip install -q --upgrade gdown albumentations imagehash opencv-python matplotlib tqdm gradio scikit-learn

import os
import gdown

DATASETS = {
    "/content/mri_dataset.zip": "1fQh-pwXNEEOsVvvEz9xUZcPfsnVnhgSs",
    "/content/not_mri_custom.zip": "1HNkX3akvRtW9Thkj8DmFeYOUpcL9CvbC",
    "/content/not_mri_nature.zip": "1fGczBDMF3QO-GE55spClLMYD1htKN7jc"
}

for dest_path, file_id in DATASETS.items():
    if not os.path.exists(dest_path):
        print(f"Загрузка {os.path.basename(dest_path)}...")
        url = f"https://drive.google.com/uc?id={file_id}"
        gdown.download(url, dest_path, quiet=False)
    else:
        print(f"Файл {os.path.basename(dest_path)} уже существует.")

In [ ]:

# ЯЧЕЙКА 2
import os
import re
import cv2
import shutil
import zipfile
import hashlib
import numpy as np
from PIL import Image
import imagehash
from sklearn.model_selection import train_test_split

RAW_DIR = "/content/raw_data"
CLEAN_DIR = "/content/clean_dataset"
DUPLICATES_DIR = "/content/duplicates"
TEMP_NOT_MRI = "/content/temp_not_mri"

ZIP_MRI = "/content/mri_dataset.zip"
ZIP_CUSTOM = "/content/not_mri_custom.zip"
ZIP_NATURE = "/content/not_mri_nature.zip"

for p in [RAW_DIR, CLEAN_DIR, DUPLICATES_DIR, TEMP_NOT_MRI]:
    if os.path.exists(p):
        shutil.rmtree(p)
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(DUPLICATES_DIR, exist_ok=True)
os.makedirs(TEMP_NOT_MRI, exist_ok=True)

def safe_extract(zip_path, extract_to):
    with zipfile.ZipFile(zip_path, 'r') as zf:
        for member in zf.infolist():
            if "__macosx" in member.filename.lower() or member.filename.split('/')[-1].startswith('._'):
                continue
            try:
                try:
                    filename = member.filename.encode('cp437').decode('cp866')
                except:
                    filename = member.filename.encode('cp437').decode('utf-8')
            except:
                filename = member.filename

            clean_path = re.sub(r'[*?"<>|]', '', filename).replace(':', '_')
            parts = [p.strip() for p in clean_path.split('/') if p.strip()]

            safe_parts = []
            for i, part in enumerate(parts):
                if len(part.encode('utf-8')) > 60:
                    if i == len(parts) - 1 and '.' in part:
                        name, ext = os.path.splitext(part)
                        part = f"f_{hashlib.md5(part.encode()).hexdigest()[:10]}{ext[:5]}"
                    else:
                        part = f"d_{hashlib.md5(part.encode()).hexdigest()[:10]}"
                safe_parts.append(part)

            if not safe_parts:
                continue
            target_path = os.path.join(extract_to, *safe_parts)
            if member.is_dir() or member.filename.endswith('/'):
                os.makedirs(target_path, exist_ok=True)
            else:
                os.makedirs(os.path.dirname(target_path), exist_ok=True)
                with zf.open(member) as source, open(target_path, "wb") as target:
                    target.write(source.read())

print("[1/5] Распаковка архива МРТ...")
safe_extract(ZIP_MRI, RAW_DIR)

for root, dirs, files in os.walk(RAW_DIR):
    for f in files:
        if f.lower().endswith('.zip') and not f.startswith('._'):
            full_p = os.path.join(root, f)
            try:
                safe_extract(full_p, root)
                os.remove(full_p)
            except Exception:
                pass

def is_valid_mri_quality(image_path):
    try:
        img_gray = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if img_gray is None or img_gray.size == 0:
            return False
        laplacian_var = cv2.Laplacian(img_gray, cv2.CV_64F).var()
        if laplacian_var < 15.0:
            return False
        hist = cv2.calcHist([img_gray], [0], None, [256], [0, 256])
        hist_norm = hist.ravel() / (hist.sum() + 1e-7)
        hist_norm = hist_norm[hist_norm > 0]
        entropy = -np.sum(hist_norm * np.log2(hist_norm))
        if entropy < 2.5:
            return False
        return True
    except:
        return False

seen_hashes = []
duplicates_found = 0
def is_duplicate_and_register(image_path, hash_threshold=2):
    global duplicates_found
    try:
        with Image.open(image_path).convert('L') as img:
            variants = [
                imagehash.phash(img),
                imagehash.phash(img.rotate(90)),
                imagehash.phash(img.rotate(180)),
                imagehash.phash(img.rotate(270)),
                imagehash.phash(img.transpose(Image.FLIP_LEFT_RIGHT))
            ]
            for h in variants:
                for exist_h, exist_path in seen_hashes:
                    if h - exist_h <= hash_threshold:
                        duplicates_found += 1
                        shutil.move(image_path, os.path.join(DUPLICATES_DIR, f"dup_{duplicates_found}_{os.path.basename(image_path)}"))
                        return True
            seen_hashes.append((variants[0], image_path))
            return False
    except:
        return True

CLASS_CANONICAL = {
    "глиома": "Глиома", "glioma": "Глиома",
    "менингиома": "Менингиома", "meningioma": "Менингиома",
    "гипофиз": "Гипофиз", "pituitary": "Гипофиз",
    "опухоли нет": "Опухоли нет", "здоровые": "Опухоли нет",
    "notumor": "Опухоли нет", "no_tumor": "Опухоли нет"
}

print("[2/5] Фильтрация шумов и дедупликация МРТ...")
discovered_classes = {}
total_filtered_noise = 0

for root, dirs, files in os.walk(RAW_DIR):
    folder_name = os.path.basename(root).strip().lower()
    if folder_name in ['raw_data', 'train', 'val', 'test', 'тестова выборка', 'обучающая выборка', 'not_mri', 'не мрт']:
        continue

    canonical_name = None
    for k, v in CLASS_CANONICAL.items():
        if k in folder_name:
            canonical_name = v
            break
    if canonical_name is None:
        continue

    for f in files:
        if f.lower().endswith(('.jpg', '.jpeg', '.png')) and not f.startswith('._'):
            full_path = os.path.join(root, f)
            if not is_valid_mri_quality(full_path):
                total_filtered_noise += 1
                continue
            if not is_duplicate_and_register(full_path):
                discovered_classes.setdefault(canonical_name, []).append(full_path)

print(f"Удалено дубликатов: {duplicates_found}")
print(f"Отфильтровано зашумленных снимков: {total_filtered_noise}")

valid_not_mri = []
TARGET_NOT_MRI_COUNT = 2000

print("[3/5] Формирование контрольного класса 'Не МРТ'...")
if os.path.exists(ZIP_NATURE):
    with zipfile.ZipFile(ZIP_NATURE, 'r') as zf:
        for m in zf.infolist():
            if len(valid_not_mri) >= TARGET_NOT_MRI_COUNT:
                break
            if m.is_dir() or "__macosx" in m.filename.lower() or '._' in m.filename:
                continue
            if m.filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                ext = os.path.splitext(m.filename)[1].lower()
                target_f = os.path.join(TEMP_NOT_MRI, f"nature_{len(valid_not_mri)}{ext}")
                with zf.open(m) as src, open(target_f, "wb") as dst:
                    dst.write(src.read())
                valid_not_mri.append(target_f)

if len(valid_not_mri) < TARGET_NOT_MRI_COUNT and os.path.exists(ZIP_CUSTOM):
    with zipfile.ZipFile(ZIP_CUSTOM, 'r') as zf:
        for m in zf.infolist():
            if len(valid_not_mri) >= TARGET_NOT_MRI_COUNT:
                break
            if m.is_dir() or "__macosx" in m.filename.lower() or '._' in m.filename:
                continue
            if m.filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                ext = os.path.splitext(m.filename)[1].lower()
                target_f = os.path.join(TEMP_NOT_MRI, f"custom_{len(valid_not_mri)}{ext}")
                with zf.open(m) as src, open(target_f, "wb") as dst:
                    dst.write(src.read())
                valid_not_mri.append(target_f)

discovered_classes["Не МРТ"] = valid_not_mri
print(f"Сформирован класс 'Не МРТ': {len(valid_not_mri)}")

print("[4/5] Разделение на train и val...")
for split in ['train', 'val']:
    os.makedirs(os.path.join(CLEAN_DIR, split), exist_ok=True)
    print("\nСтруктура датасета:")
for c_name, c_files in sorted(discovered_classes.items()):
    train_f, val_f = train_test_split(c_files, test_size=0.2, random_state=42, shuffle=True)
    os.makedirs(os.path.join(CLEAN_DIR, 'train', c_name), exist_ok=True)
    os.makedirs(os.path.join(CLEAN_DIR, 'val', c_name), exist_ok=True)

    for idx, src in enumerate(train_f):
        ext = os.path.splitext(src)[1]
        shutil.copy(src, os.path.join(CLEAN_DIR, 'train', c_name, f"img_{idx}{ext}"))
    for idx, src in enumerate(val_f):
        ext = os.path.splitext(src)[1]
        shutil.copy(src, os.path.join(CLEAN_DIR, 'val', c_name, f"img_{idx}{ext}"))

    print(f" - {c_name}: всего {len(c_files)} (train: {len(train_f)}, val: {len(val_f)})")

shutil.rmtree(TEMP_NOT_MRI)
print("\n[5/5] Подготовка датасета завершена.")

In [ ]:

# ЯЧЕЙКА 3
import os
import cv2
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler
from torchvision import transforms, models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGET_IMAGE_SIZE = (512, 512)

class MedicalPreprocess:
    def __init__(self, target_size=TARGET_IMAGE_SIZE):
        self.target_size = target_size
        self.clahe = cv2.createCLAHE(clipLimit=1.2, tileGridSize=(16, 16))

    def __call__(self, img_pil):
        img = np.array(img_pil)

        if len(img.shape) == 2:
            gray = img
        elif len(img.shape) == 3 and img.shape[2] == 3:
            gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        else:
            gray = img[:, :, 0]

        _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            c = max(contours, key=cv2.contourArea)
            if cv2.contourArea(c) > 1000:
                x, y, w, h = cv2.boundingRect(c)
                img = img[y:y+h, x:x+w]
                gray = gray[y:y+h, x:x+w]

        enhanced = self.clahe.apply(gray)
        img = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2RGB)

        h, w = img.shape[:2]
        target_w, target_h = self.target_size
        scale = min(target_w / w, target_h / h)
        nw, nh = max(1, int(w * scale)), max(1, int(h * scale))

        resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
        canvas = np.zeros((target_h, target_w, 3), dtype=np.uint8)

        dx = (target_w - nw) // 2
        dy = (target_h - nh) // 2
        canvas[dy:dy+nh, dx:dx+nw] = resized

        return Image.fromarray(canvas)

data_transforms = {
    'train': transforms.Compose([
        MedicalPreprocess(target_size=TARGET_IMAGE_SIZE),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.30, scale=(0.02, 0.15), value=0)
    ]),
    'val': transforms.Compose([
        MedicalPreprocess(target_size=TARGET_IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

class GatekeeperDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        self.classes = ['Не МРТ', 'МРТ мозга']
        for root, dirs, files in os.walk(root_dir):
            folder = os.path.basename(root)
            if folder in ['train', 'val', 'clean_dataset']:
                continue
            label = 0 if folder == 'Не МРТ' else 1
            for f in files:
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(root, f), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

class DiagnosticDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        self.classes = ['Гипофиз', 'Глиома', 'Менингиома', 'Опухоли нет']
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        for root, dirs, files in os.walk(root_dir):
            folder = os.path.basename(root)
            if folder in self.class_to_idx:
                label = self.class_to_idx[folder]
                for f in files:
                    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                        self.samples.append((os.path.join(root, f), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

CLEAN_DIR = "/content/clean_dataset"
BATCH_SIZE = 16

gate_datasets = {x: GatekeeperDataset(os.path.join(CLEAN_DIR, x), data_transforms[x]) for x in ['train', 'val']}
gate_loaders = {x: DataLoader(gate_datasets[x], batch_size=BATCH_SIZE, shuffle=(x == 'train'), num_workers=2, pin_memory=True) for x in ['train', 'val']}

diag_datasets = {x: DiagnosticDataset(os.path.join(CLEAN_DIR, x), data_transforms[x]) for x in ['train', 'val']}
diag_loaders = {x: DataLoader(diag_datasets[x], batch_size=BATCH_SIZE, shuffle=(x == 'train'), num_workers=2, pin_memory=True) for x in ['train', 'val']}

diag_classes = diag_datasets['train'].classes
gate_classes = gate_datasets['train'].classes

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1.0 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


diag_weights = np.array([1.10, 1.30, 1.15, 0.65], dtype=np.float32)
diag_weights_tensor = torch.FloatTensor(diag_weights).to(device)

model_gate = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
model_gate.classifier[3] = nn.Linear(model_gate.classifier[3].in_features, 2)
model_gate = model_gate.to(device)
criterion_gate = nn.CrossEntropyLoss()
optimizer_gate = torch.optim.AdamW(model_gate.parameters(), lr=3e-4, weight_decay=1e-3)

model_diag = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model_diag.fc = nn.Sequential(
    nn.Dropout(0.30),
    nn.Linear(model_diag.fc.in_features, len(diag_classes))
)
model_diag = model_diag.to(device)
criterion_diag = FocalLoss(alpha=diag_weights_tensor, gamma=2.0)
scaler = GradScaler()

print(f"Инициализация завершена. Разрешение: {TARGET_IMAGE_SIZE}, Батч: {BATCH_SIZE}")

In [ ]:

# ЯЧЕЙКА 4

import copy
import warnings
from sklearn.metrics import classification_report, f1_score

warnings.filterwarnings("ignore")
ACCUMULATION_STEPS = 2


print("Обучение Gatekeeper (3 эпохи)...")
best_gate_acc = 0.0
best_gate_wts = copy.deepcopy(model_gate.state_dict())

for epoch in range(3):
    print(f"Gatekeeper: эпоха {epoch + 1}/3")
    for phase in ['train', 'val']:
        model_gate.train() if phase == 'train' else model_gate.eval()
        running_corrects = 0
        total = 0

        for inputs, labels in gate_loaders[phase]:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer_gate.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                with torch.amp.autocast('cuda'):
                    outputs = model_gate(inputs)
                    loss = criterion_gate(outputs, labels)
                _, preds = torch.max(outputs, 1)

                if phase == 'train':
                    scaler.scale(loss).backward()
                    scaler.step(optimizer_gate)
                    scaler.update()

            running_corrects += torch.sum(preds == labels.data).item()
            total += labels.size(0)

        acc = running_corrects / total
        print(f"Gatekeeper {phase.upper()} Точность: {acc:.4f}")
        if phase == 'val' and acc > best_gate_acc:
            best_gate_acc = acc
            best_gate_wts = copy.deepcopy(model_gate.state_dict())

model_gate.load_state_dict(best_gate_wts)
torch.save(best_gate_wts, 'best_gatekeeper.pth')
print(f"Gatekeeper сохранен с точностью: {best_gate_acc:.4f}\n")


print("Диагностика: Фаза 1 (замороженный backbone, 3 эпохи)...")
for param in model_diag.parameters():
    param.requires_grad = False
for param in model_diag.fc.parameters():
    param.requires_grad = True

optimizer_head = torch.optim.AdamW(model_diag.fc.parameters(), lr=3e-4, weight_decay=1e-3)

for epoch in range(3):
    model_diag.train()
    running_loss = 0.0
    optimizer_head.zero_grad()
    for batch_idx, (inputs, labels) in enumerate(diag_loaders['train']):
        inputs, labels = inputs.to(device), labels.to(device)
        with torch.amp.autocast('cuda'):
            outputs = model_diag(inputs)
            loss = criterion_diag(outputs, labels) / ACCUMULATION_STEPS
        scaler.scale(loss).backward()
        if (batch_idx + 1) % ACCUMULATION_STEPS == 0 or (batch_idx + 1) == len(diag_loaders['train']):
            scaler.step(optimizer_head)
            scaler.update()
            optimizer_head.zero_grad()
        running_loss += loss.item() * inputs.size(0) * ACCUMULATION_STEPS
    print(f"Фаза 1: эпоха {epoch + 1}/3 | Потери: {running_loss / len(diag_datasets['train']):.4f}")


print("\nДиагностика: Фаза 2 (глубокая доводка всей сети, до 25 эпох)...")
for param in model_diag.parameters():
    param.requires_grad = True

backbone_params = [p for n, p in model_diag.named_parameters() if not n.startswith('fc')]
classifier_params = model_diag.fc.parameters()

optimizer_full = torch.optim.AdamW([
    {'params': backbone_params, 'lr': 1.2e-5},
    {'params': classifier_params, 'lr': 8e-5}
], weight_decay=1e-3)

scheduler_full = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer_full, T_0=5, T_mult=2, eta_min=1e-7
)

best_diag_f1 = 0.0
best_diag_wts = copy.deepcopy(model_diag.state_dict())
patience = 7
patience_counter = 0

for epoch in range(25):
    print(f"\nДиагностика Фаза 2: эпоха {epoch + 1}/25")
    for phase in ['train', 'val']:
        model_diag.train() if phase == 'train' else model_diag.eval()
        running_loss = 0.0
        running_corrects = 0
        all_preds, all_labels = [], []
        optimizer_full.zero_grad()

        for batch_idx, (inputs, labels) in enumerate(diag_loaders[phase]):
            inputs, labels = inputs.to(device), labels.to(device)

            with torch.set_grad_enabled(phase == 'train'):
                with torch.amp.autocast('cuda'):
                    outputs = model_diag(inputs)
                    loss = criterion_diag(outputs, labels)
                    loss_scaled = loss / ACCUMULATION_STEPS
                _, preds = torch.max(outputs, 1)

                if phase == 'train':
                    scaler.scale(loss_scaled).backward()
                    if (batch_idx + 1) % ACCUMULATION_STEPS == 0 or (batch_idx + 1) == len(diag_loaders[phase]):
                        scaler.step(optimizer_full)
                        scaler.update()
                        optimizer_full.zero_grad()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

            if phase == 'val':
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        if phase == 'train':
            scheduler_full.step()

        epoch_loss = running_loss / len(diag_datasets[phase])
        epoch_acc = running_corrects.double() / len(diag_datasets[phase])
        print(f"Диагностика {phase.upper()} Потери: {epoch_loss:.4f} | Точность: {epoch_acc:.4f}")

        if phase == 'val':
            macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
            print(f"Валидация Macro F1: {macro_f1:.4f}")
            print(classification_report(all_labels, all_preds, target_names=diag_classes, zero_division=0))

            if macro_f1 > best_diag_f1:
                best_diag_f1 = macro_f1
                best_diag_wts = copy.deepcopy(model_diag.state_dict())
                torch.save(best_diag_wts, 'best_resnet50_mri_fixed.pth')
                if os.path.exists('/content/drive/MyDrive'):
                    torch.save(best_diag_wts, '/content/drive/MyDrive/best_resnet50_mri_fixed.pth')
                print(f"Сохранен новый рекорд Macro F1: {best_diag_f1:.4f}")
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print("Остановка по критерию стабилизации метрики.")
                    break

model_diag.load_state_dict(best_diag_wts)
print(f"\nОбучение завершено. Лучший итоговый Macro F1: {best_diag_f1:.4f}")

In [ ]:

# ЯЧЕЙКА 4.1
import copy
import torch
import torch.nn as nn

criterion_gate = nn.CrossEntropyLoss(weight=torch.FloatTensor([1.5, 1.0]).to(device))
optimizer_gate = torch.optim.AdamW(model_gate.parameters(), lr=2e-4, weight_decay=1e-3)

print("Обучение Gatekeeper (MobileNetV3)...")
best_gate_acc = 0.0
best_gate_wts = copy.deepcopy(model_gate.state_dict())

for epoch in range(2):
    for phase in ['train', 'val']:
        model_gate.train() if phase == 'train' else model_gate.eval()
        running_corrects, total = 0, 0

        for inputs, labels in gate_loaders[phase]:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer_gate.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                with torch.amp.autocast('cuda'):
                    outputs = model_gate(inputs)
                    loss = criterion_gate(outputs, labels)
                _, preds = torch.max(outputs, 1)

                if phase == 'train':
                    scaler.scale(loss).backward()
                    scaler.step(optimizer_gate)
                    scaler.update()

            running_corrects += torch.sum(preds == labels.data).item()
            total += labels.size(0)

        acc = running_corrects / total
        print(f"Gatekeeper Эпоха {epoch + 1}/2 | {phase.upper()} Точность: {acc:.4f}")
        if phase == 'val' and acc > best_gate_acc:
            best_gate_acc = acc
            best_gate_wts = copy.deepcopy(model_gate.state_dict())

model_gate.load_state_dict(best_gate_wts)
torch.save(best_gate_wts, 'best_gatekeeper.pth')
print(f"Gatekeeper сохранен с точностью: {best_gate_acc:.4f}")